# NB6 — Table Maintenance: the 4 jobs that are not optional

**Stack:** `deltalake` + `pyiceberg`. Maps to slide §6 (Storage Optimization →
*Table Maintenance: 4 Job Bắt Buộc*) + §12 FinOps + deliverable bullet 6.

> **The small-file problem is the single most common production failure mode
> of a lakehouse** — more common than every other cause combined. It is not
> caused by bad code. It is caused by *normal* streaming ingestion plus the
> absence of a cron job.

The four mandatory jobs, per the slide:

| # | Job | If you skip it | Delta | Iceberg |
|---|---|---|---|---|
| 1 | Compaction | Small files → slow queries, non-linear cost | `OPTIMIZE` | `rewrite_data_files` |
| 2 | Clustering / sort | Loose stats → no file skipping | `ZORDER` / Liquid | `sort` / `zorder` |
| 3 | Snapshot expiry | Metadata bloat, slow listing, storage garbage | `VACUUM RETAIN` | `expire_snapshots` |
| 4 | Orphan removal | You pay for invisible garbage from failed jobs | `VACUUM` | `remove_orphan_files` |

You will run all four and **measure** each one.

In [17]:
import _setup  # noqa: F401

import datetime as dtm
import glob
import os
import time
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import DeltaTable, write_deltalake

from lakehouse import catalog, count_files, du, human, namespace, path, reset, reset_catalog

TABLE = path("scratch", "maint_events")
reset(TABLE)

## 0. Manufacture the problem: 200 micro-batches

This is what a Kafka→lakehouse job with a 5-second trigger does to you
overnight. Each commit is small and perfectly correct. The *accumulation*
is the bug.

In [18]:
import random

N_BATCHES, ROWS_PER_BATCH = 200, 500
USERS = 20_000
random.seed(42)
ALPHABET = "abcdefghijklmnopqrstuvwxyz0123456789"

t0 = time.perf_counter()
for b in range(N_BATCHES):
    batch = pa.table({
        "event_id":   [b * ROWS_PER_BATCH + i for i in range(ROWS_PER_BATCH)],
        "user_id":    [(b * 7919 + i * 31) % USERS for i in range(ROWS_PER_BATCH)],
        "ts":         [dtm.datetime(2026, 8, 1 + b % 10, b % 24, i % 60) for i in range(ROWS_PER_BATCH)],
        "latency_ms": [200 + (i * 13) % 2500 for i in range(ROWS_PER_BATCH)],
        # A prompt/response blob. Without a wide column the rows are so tiny
        # that everything compacts into one file and there is nothing to skip.
        "payload":    ["".join(random.choices(ALPHABET, k=80)) for _ in range(ROWS_PER_BATCH)],
    })
    write_deltalake(TABLE, batch, mode="append" if b else "overwrite")
ingest_s = time.perf_counter() - t0


def snapshot_metrics(label: str) -> dict:
    """The four numbers that actually matter. Deterministic — not wall-clock."""
    dt = DeltaTable(TABLE)
    log = Path(TABLE) / "_delta_log"
    m = {
        "data files":     len(dt.file_uris()),
        "data bytes":     du(TABLE) - du(log),
        "log bytes":      du(log),
        "log entries":    len(list(log.glob("*.json"))),
        "rows":           dt.count(),
    }
    print(f"{label:24s} files={m['data files']:>4}  data={human(m['data bytes']):>9}  "
          f"log={human(m['log bytes']):>9} ({m['log entries']} json)  rows={m['rows']:,}")
    return m


print(f"Ingested {N_BATCHES * ROWS_PER_BATCH:,} rows in {N_BATCHES} commits ({ingest_s:.1f}s)\n")
base = snapshot_metrics("BASELINE (the mess)")
print(f"\nAverage file size: {human(base['data bytes'] / base['data files'])}"
      "   ← production target is 128–512 MB")

Ingested 100,000 rows in 200 commits (22.8s)

BASELINE (the mess)      files= 200  data=  10.1 MB  log= 385.7 KB (200 json)  rows=100,000

Average file size: 51.5 KB   ← production target is 128–512 MB


### 📝 Nhận xét — Tái hiện small-file problem

- 200 commit × 500 dòng = 100.000 dòng → **200 file Parquet**, tổng ~10,1 MB, trung bình **~51,5 KB/file** (production nhắm 128–512 MB/file — nhỏ hơn ~2.500–10.000 lần), và **200 file JSON** trong `_delta_log/`.
- Mỗi commit đều **đúng**; lỗi nằm ở sự **tích luỹ**: streaming với trigger vài giây + không có job bảo trì. Đây là failure mode phổ biến nhất của lakehouse, không do code sai.
- `snapshot_metrics` đo bằng **số file / bytes / số entry log** — các con số tất định, không phụ thuộc thời gian chạy như NB2.


### Why small files cost non-linearly

Object storage charges per **request**, not just per byte, and every file
costs a `GET` (plus metadata to plan over). 200 files is invisible;
200 million files is a platform outage.

In [19]:
GET_PER_1K = 0.0004     # S3 GET list price, USD per 1,000 requests
QUERIES_PER_DAY = 50_000
gets_now = base["data files"] * QUERIES_PER_DAY
print(f"Full-scan GETs/day at {base['data files']} files: {gets_now:,}"
      f"  →  ${gets_now / 1000 * GET_PER_1K:,.2f}/day just in requests")
print(f"Same data in 4 compacted files:      {4 * QUERIES_PER_DAY:,}"
      f"  →  ${4 * QUERIES_PER_DAY / 1000 * GET_PER_1K:,.2f}/day")

Full-scan GETs/day at 200 files: 10,000,000  →  $4.00/day just in requests
Same data in 4 compacted files:      200,000  →  $0.08/day


### 📝 Nhận xét — Vì sao file nhỏ tốn tiền phi tuyến

- Object storage tính tiền **theo request**, không chỉ theo byte: mỗi file là ít nhất một `GET` (chưa kể đọc footer, metadata để lập kế hoạch).
- 200 file × 50.000 full-scan/ngày = **10 triệu GET ≈ $4/ngày**; cùng dữ liệu trong 4 file chỉ **$0,08/ngày** (ít hơn 50×). Lượng dữ liệu không đổi — chỉ số file thay đổi.
- Ở quy mô 200 file con số này vô hình; ở 200 triệu file nó thành sự cố nền tảng (listing chậm, planning chậm, hoá đơn request vượt hoá đơn lưu trữ).


## Job 1 — Compaction

`OPTIMIZE` rewrites many small files into few large ones. It is a *new
commit*: the old files are not deleted, they are **tombstoned** (that
matters for Job 3).

In [20]:
# Lab scale: 1 MB target. Production targets 128–512 MB — the *ratio* of
# before:after is the lesson, not the absolute size.
TARGET_SIZE = 1024 * 1024

dt = DeltaTable(TABLE)
metrics = dt.optimize.compact(target_size=TARGET_SIZE)
after_compact = snapshot_metrics("AFTER compaction")

print(f"\nfilesAdded={metrics['numFilesAdded']}  filesRemoved={metrics['numFilesRemoved']}")
print(f"File reduction: {base['data files']} → {after_compact['data files']} "
      f"({base['data files'] / max(after_compact['data files'], 1):.0f}× fewer)")
print("Note: data bytes went UP slightly — compaction WRITES new files before")
print("the old ones are reclaimed. You pay twice, briefly. Budget for it.")

AFTER compaction         files=  11  data=  16.1 MB  log= 434.4 KB (201 json)  rows=100,000

filesAdded=11  filesRemoved=200
File reduction: 200 → 11 (18× fewer)
Note: data bytes went UP slightly — compaction WRITES new files before
the old ones are reclaimed. You pay twice, briefly. Budget for it.


### 📝 Nhận xét — Job 1: Compaction

- `compact(target_size=1MB)` gom 200 file → **11 file (18× ít hơn, ≥ 10× ✅)**: `filesAdded=11`, `filesRemoved=200`.
- OPTIMIZE là **một commit mới** (log 200 → 201 JSON). 200 file cũ **không bị xoá** mà chỉ bị đánh dấu `remove` (**tombstone**) — vẫn nằm trên đĩa để time travel còn đọc được.
- Vì vậy dung lượng trên đĩa **tạm thời tăng** 10,1 → 16,1 MB (~+60%, notebook gọi là "slightly" nhưng thực tế không nhỏ): file mới và file cũ cùng tồn tại cho đến khi VACUUM (Job 3). Phải dự trù dung lượng và chi phí cho giai đoạn "trả tiền hai lần" này.
- Sau Job 3, 10 file hiện hành chỉ còn **~6,2 MB — nhỏ hơn ~40% so với 10,1 MB ban đầu**: file lớn nén tốt hơn, và delta-rs ghi file compact bằng **zstd** (tên `…zstd.parquet`) thay vì snappy của các lần append.
- Compaction **không sắp xếp** dữ liệu → chỉ giảm chi phí mỗi file, chưa giúp bỏ qua file (xem Job 2).


## Job 2 — Clustering, measured by *stats quality* (not by stopwatch)

NB2 timed Z-ORDER. Timing is noisy on a laptop, so here we measure the thing
timing is a proxy for: **how many files could possibly contain the row you
want**, according to per-file min/max stats in the log.

That count is what the engine skips on. It is deterministic.

In [21]:
TARGET_USER = 12_345


def files_touched(target: int = TARGET_USER) -> int:
    """Files whose [min,max] user_id range could contain `target`."""
    # delta-rs 1.x returns an arro3 Table; pa.table() adopts it via the
    # Arrow C stream interface (zero copy, no pandas needed).
    aa = pa.table(DeltaTable(TABLE).get_add_actions(flatten=True)).to_pylist()
    return sum(1 for f in aa if f["min.user_id"] <= target <= f["max.user_id"])


before_cluster = files_touched()
DeltaTable(TABLE).optimize.z_order(["user_id"], target_size=TARGET_SIZE)
after_cluster = files_touched()
total_files = len(DeltaTable(TABLE).file_uris())

print(f"Point query user_id={TARGET_USER}")
print(f"  before clustering: must open {before_cluster}/{after_compact['data files']} files")
print(f"  after  clustering: must open {after_cluster}/{total_files} files")
print(f"  → skip rate: {(1 - after_cluster / max(total_files, 1)) * 100:.0f}% of files never touched")
print("\nUnclustered data has overlapping min/max ranges, so stats prove nothing")
print("and the engine must read everything. Clustering is what makes stats USEFUL.")

Point query user_id=12345
  before clustering: must open 11/11 files
  after  clustering: must open 1/10 files
  → skip rate: 90% of files never touched

Unclustered data has overlapping min/max ranges, so stats prove nothing
and the engine must read everything. Clustering is what makes stats USEFUL.


### 📝 Nhận xét — Job 2: Clustering đo bằng chất lượng stats

- Thay vì bấm giờ (nhiễu, như NB2), ta đếm **số file có khoảng `[min, max]` của `user_id` chứa 12345** — đúng con số engine dùng để quyết định bỏ qua file, và **tất định**.
- Sau compaction: **11/11 file** đều "có thể chứa" 12345 → phải mở tất cả. Compaction chỉ gom file, không sắp xếp, nên mỗi file vẫn phủ gần toàn bộ dải `user_id`.
- Sau `z_order(["user_id"])`: chỉ **1/10 file** → **skip 90%** (≥ 50% ✅). Clustering làm khoảng min/max của từng file hẹp và gần như không chồng lấn ⇒ stats trong log trở nên **hữu ích**.
- Kết luận Job 1 + Job 2: compaction giảm **chi phí mỗi file**, clustering giảm **số file phải đọc** — cần cả hai.


## Job 3 — Snapshot / version expiry

Every commit so far is still fully readable — that's time travel, and it is
not free. Those tombstoned files are still on disk and still billed.

⚠️ **`retention_hours=0` destroys your ability to time-travel and can break
readers mid-query.** Production uses ≥ 168h (7 days). We force it here only
to make the reclaim visible inside a lab.

In [22]:
dt = DeltaTable(TABLE)
doomed = dt.vacuum(retention_hours=0, dry_run=True, enforce_retention_duration=False)
print(f"VACUUM would reclaim {len(doomed)} tombstoned files "
      f"({human(sum(du(f) for f in doomed))})")

before_vacuum = du(TABLE)
dt.vacuum(retention_hours=0, dry_run=False, enforce_retention_duration=False)
after_vacuum = snapshot_metrics("AFTER vacuum")
print(f"\nReclaimed: {human(before_vacuum - du(TABLE))}")
print(f"Time travel to v0 is now GONE — that is the trade you just made.")

VACUUM would reclaim 211 tombstoned files (0 B)


AFTER vacuum             files=  10  data=   6.2 MB  log= 445.1 KB (204 json)  rows=100,000

Reclaimed: 16.1 MB
Time travel to v0 is now GONE — that is the trade you just made.


### 📝 Nhận xét — Job 3: VACUUM (version expiry)

- VACUUM xoá vật lý các file **đã bị tombstone và quá hạn retention**: **211 file** = 200 file gốc (bị compaction thay thế) + 11 file của compaction (bị Z-order thay thế).
- Thu hồi **16,1 MB**; data còn **6,2 MB** với đúng 10 file hiện hành, `rows=100,000` không đổi ⇒ dữ liệu hiện tại còn nguyên ✅.
- ⚠️ **Đọc output cẩn thận:** dòng `VACUUM would reclaim 211 tombstoned files (0 B)` — **"0 B" là lỗi hiển thị**: `vacuum(dry_run=True)` trả về **đường dẫn tương đối** (`part-00000-….parquet`), nên `du()` không tìm thấy file và trả 0. Số bytes thật là dòng `Reclaimed: 16.1 MB` (đo bằng `du(TABLE)` trước/sau).
- **Đánh đổi:** `retention_hours=0` làm **time travel về các version cũ không còn khả dụng** (file của chúng đã bị xoá) và có thể làm hỏng reader đang đọc dở. Production dùng ≥ 168h (7 ngày). Liên hệ NB3: **cửa sổ RESTORE chính là cửa sổ retention** — chỉ dùng retention 0 với dữ liệu scratch của lab.
- Log tăng lên 204 JSON vì VACUUM cũng ghi commit (`VACUUM START` / `VACUUM END`) để audit.


## Job 4 — Orphan files: the garbage nobody can see

An orphan is a file written by a job that **crashed before committing**. It
is on disk, you are billed for it, and *no metadata references it* — so it
is invisible to `history()`, to `files()`, and to your dashboards.

We simulate three crashed writers:

In [23]:
for i in range(3):
    orphan = Path(TABLE) / f"part-9999{i}-crashed-writer-c000.snappy.parquet"
    pq.write_table(pa.table({"event_id": list(range(1000)), "user_id": [0] * 1000,
                             "ts": [dtm.datetime(2026, 8, 1)] * 1000,
                             "latency_ms": [1] * 1000,
                             "payload": ["x" * 80] * 1000}), orphan)
    old = time.time() - 30 * 86400          # 30 days old — well past any retention
    os.utime(orphan, (old, old))

dt = DeltaTable(TABLE)
print(f"Rows reported by the table: {dt.count():,}   (orphans are invisible)")
print(f"Parquet files on disk:      {count_files(TABLE)}")
print(f"Parquet files in the log:   {len(dt.file_uris())}")
print(f"→ {count_files(TABLE) - len(dt.file_uris())} files you pay for and cannot see")

Rows reported by the table: 100,000   (orphans are invisible)
Parquet files on disk:      15
Parquet files in the log:   10
→ 5 files you pay for and cannot see


### 📝 Nhận xét — Job 4: Orphan — rác không ai nhìn thấy

- 3 file `part-9999x-crashed-writer` mô phỏng writer **crash trước khi commit**: file nằm trên đĩa, bị tính tiền, nhưng **không metadata nào tham chiếu** → `count()` vẫn 100.000 dòng, `file_uris()` vẫn 10 file.
- ⚠️ **Đọc output cẩn thận:** `Parquet files on disk: 15 … → 5 files you pay for` — thực tế chỉ có **3 orphan**. 2 file còn lại là **checkpoint delta-rs tự ghi** mỗi 100 commit (`_delta_log/…099.checkpoint.parquet`, `…199.checkpoint.parquet`); `count_files` đếm đệ quy nên tính cả `_delta_log/`. Bằng chứng: sau khi xoá orphan vẫn còn "on disk 12, in log 10" (12 − 10 = 2 checkpoint).


### Measured finding: `VACUUM` alone does **not** catch these

Run vacuum again on a table whose orphans are 30 days old:

In [24]:
still = DeltaTable(TABLE).vacuum(retention_hours=0, dry_run=True, enforce_retention_duration=False)
print(f"VACUUM dry-run now finds: {len(still)} files")
print(f"Orphans still on disk:    {count_files(TABLE) - len(DeltaTable(TABLE).file_uris())}")
print("""
`deltalake` (the Rust/Python implementation used here) reclaims files the
transaction log has TOMBSTONED. A file that was never committed was never
tombstoned, so the log has no idea it exists.

Spark's VACUUM additionally lists the table directory, which is why the slide
lists VACUUM under orphan removal — but you should never *assume* your engine
does the directory pass. Verify it, or run the diff yourself:
""")

VACUUM dry-run now finds: 211 files
Orphans still on disk:    5

`deltalake` (the Rust/Python implementation used here) reclaims files the
transaction log has TOMBSTONED. A file that was never committed was never
tombstoned, so the log has no idea it exists.

Spark's VACUUM additionally lists the table directory, which is why the slide
lists VACUUM under orphan removal — but you should never *assume* your engine
does the directory pass. Verify it, or run the diff yourself:



### 📝 Nhận xét — VACUUM không bắt được orphan

- Dù orphan đã 30 ngày tuổi (vượt mọi retention), VACUUM **không xoá chúng**. `deltalake` (delta-rs) chỉ thu hồi file **đã bị tombstone trong log**; file chưa từng commit thì chưa từng bị tombstone ⇒ log không biết nó tồn tại.
- ⚠️ **Đọc output cẩn thận:** `VACUUM dry-run now finds: 211 files` — đó là **211 file đã bị xoá ở Job 3** (tombstone vẫn còn trong log nên dry-run vẫn liệt kê; kiểm tra thực tế 0/211 file còn trên đĩa). **Không file orphan nào** nằm trong danh sách — đúng điều cần chứng minh.
- Spark VACUUM có thêm bước liệt kê thư mục nên bắt được orphan — vì vậy slide xếp VACUUM vào orphan removal. Bài học: **đừng giả định engine làm bước liệt kê thư mục — phải kiểm chứng** với đúng engine/phiên bản mình dùng. Hành vi này được "ghim" bằng test canary trong `tests/`.


### The orphan-removal algorithm (this is all `remove_orphan_files` does)

Set difference: *files on disk* − *files referenced by live metadata*, with
an age guard so you never delete a file an in-flight writer is still using.

In [25]:
def find_orphans(table_path: str, min_age_hours: int = 24) -> list[str]:
    """Files on disk that no live snapshot references, older than the guard."""
    referenced = {os.path.realpath(u.replace("file://", ""))
                  for u in DeltaTable(table_path).file_uris()}
    cutoff = time.time() - min_age_hours * 3600
    orphans = []
    for f in Path(table_path).rglob("*.parquet"):
        if "_delta_log" in f.parts:
            continue
        if os.path.realpath(f) not in referenced and f.stat().st_mtime < cutoff:
            orphans.append(str(f))
    return orphans


found = find_orphans(TABLE)
reclaimed = sum(du(f) for f in found)
print(f"Orphans found: {len(found)}  ({human(reclaimed)})")
for f in found:
    print(f"  {os.path.basename(f)}")
    os.remove(f)

print(f"\nAfter removal — on disk: {count_files(TABLE)}, in log: {len(DeltaTable(TABLE).file_uris())}")
print("\n⚠️ The age guard is not optional. Without it you will delete files that a")
print("   concurrent writer has written but not yet committed, and corrupt the table.")

Orphans found: 3  (21.2 KB)
  part-99990-crashed-writer-c000.snappy.parquet
  part-99991-crashed-writer-c000.snappy.parquet
  part-99992-crashed-writer-c000.snappy.parquet

After removal — on disk: 12, in log: 10

⚠️ The age guard is not optional. Without it you will delete files that a
   concurrent writer has written but not yet committed, and corrupt the table.


### 📝 Nhận xét — Thuật toán dọn orphan

- `remove_orphan_files` về bản chất là **phép hiệu tập hợp**: *file trên đĩa* − *file được metadata còn sống tham chiếu* (bỏ qua `_delta_log/`).
- Tìm và xoá đúng **3 orphan (~21 KB)**; sau đó `find_orphans(TABLE) == []`.
- **Ngưỡng tuổi (`min_age_hours=24`) là bắt buộc**: một writer đang chạy đã ghi file Parquet nhưng chưa kịp commit log — file đó *trông giống hệt* orphan. Xoá nó ⇒ commit sau đó trỏ tới file không tồn tại ⇒ **hỏng bảng**. Ngưỡng tuổi phải lớn hơn thời gian chạy dài nhất của mọi writer.


## Job 5 — Manifest / log rewrite (the streaming tax)

200 commits = 200 JSON log entries. A cold reader replays **all of them** to
learn the current state. A checkpoint collapses that into one Parquet file.

In [26]:
log_dir = Path(TABLE) / "_delta_log"
json_before = len(list(log_dir.glob("*.json")))

DeltaTable(TABLE).create_checkpoint()

ckpt = list(log_dir.glob("*.checkpoint.parquet"))
print(f"JSON log entries a cold reader would replay: {json_before}")
print(f"Checkpoint written: {ckpt[0].name if ckpt else 'NONE'}")
print(f"_last_checkpoint present: {(log_dir / '_last_checkpoint').exists()}")
print("\nA reader now loads 1 checkpoint + the few JSONs after it, not all 200.")
print("For CDC/streaming tables this is the difference between a 200 ms and a")
print("20 s cold start — and it is why the slide calls it the 5th job.")

JSON log entries a cold reader would replay: 204
Checkpoint written: 00000000000000000099.checkpoint.parquet
_last_checkpoint present: True

A reader now loads 1 checkpoint + the few JSONs after it, not all 200.
For CDC/streaming tables this is the difference between a 200 ms and a
20 s cold start — and it is why the slide calls it the 5th job.


### 📝 Nhận xét — Job 5: Checkpoint

- Mỗi commit là 1 file JSON; reader khởi động lạnh phải replay log để biết trạng thái hiện tại. **Checkpoint** gộp toàn bộ trạng thái tại một version vào **1 file Parquet**, `_last_checkpoint` trỏ tới nó ⇒ reader đọc checkpoint + vài JSON sau đó.
- `_last_checkpoint present: True` ✅ — file này ghi `{"version": 203, "numOfAddFiles": 10, ...}`, tức checkpoint vừa tạo ở **version 203** (commit cuối sau VACUUM).
- ⚠️ **Đọc output cẩn thận:** dòng `Checkpoint written: 00000000000000000099.checkpoint.parquet` in **file đầu tiên mà `glob` tìm thấy** — là checkpoint delta-rs **tự động** tạo ở v99, không phải file vừa tạo (`…203.checkpoint.parquet`). `_delta_log/` hiện có 3 checkpoint: 99, 199, 203.
- Hệ quả: câu "replay all 200" không hoàn toàn đúng với delta-rs — engine đã tự checkpoint mỗi 100 commit. Job 5 vẫn quan trọng với engine/cấu hình không tự checkpoint, và với bảng CDC/streaming commit dày đặc (khởi động lạnh 200 ms so với 20 s).


## The same four jobs on Iceberg

Different spelling, identical obligations. `expire_snapshots` is the one
pyiceberg exposes natively today; compaction (`rewrite_data_files`) and
`remove_orphan_files` are Spark/Java procedures — so on the Python path you
run the diff yourself, exactly as above.

In [27]:
CAT = "nb6"          # own catalog dir — see scripts/lakehouse.py:_catalog_dir
reset_catalog(CAT)
cat = catalog(CAT)
ns = namespace(cat, "lake")

ICE_SCHEMA = pa.schema([
    pa.field("event_id", pa.int64(), nullable=False),
    pa.field("user_id", pa.int64()),
])
ice = cat.create_table(f"{ns}.maint", schema=ICE_SCHEMA)
for b in range(20):
    ice.append(pa.table({"event_id": list(range(b * 100, (b + 1) * 100)),
                         "user_id": [(b * 31 + i) % 5000 for i in range(100)]}, schema=ICE_SCHEMA))
ice = cat.load_table(f"{ns}.maint")

ice_loc = ice.location().replace("file://", "")
ice_meta = Path(ice_loc) / "metadata"


def ice_metrics(label: str) -> dict:
    t = cat.load_table(f"{ns}.maint")
    m = {
        "snapshots":      len(t.snapshots()),
        "manifest avro":  len(list(ice_meta.glob("*.avro"))),
        "metadata json":  len(list(ice_meta.glob("*.json"))),
        "metadata bytes": du(str(ice_meta)),
    }
    print(f"{label:14s} snapshots={m['snapshots']:>3}  avro={m['manifest avro']:>3}  "
          f"json={m['metadata json']:>3}  metadata={human(m['metadata bytes'])}")
    return m


ice_before = ice_metrics("before expiry")

before expiry  snapshots= 20  avro= 40  json= 21  metadata=350.5 KB


### 📝 Nhận xét — Iceberg: trạng thái trước expiry

- 20 lần `append` → **20 snapshot**, **40 file avro** (mỗi snapshot = 1 manifest list `snap-*.avro` + 1 manifest mới) và **21 metadata.json**.
- PyIceberg hiện chỉ có sẵn `expire_snapshots`; `rewrite_data_files` (compaction) và `remove_orphan_files` là procedure của Spark/Java ⇒ trên đường Python phải tự làm phép hiệu tập hợp như Job 4.


### `expire_snapshots` — keep the newest, drop the rest

Watch the avro count, not just the snapshot count.

In [28]:
KEEP_LAST = 3
doomed_ids = [s.snapshot_id for s in ice.snapshots()[:-KEEP_LAST]]
ice.maintenance.expire_snapshots().by_ids(doomed_ids).commit()
ice = cat.load_table(f"{ns}.maint")

ice_after = ice_metrics("after expiry")
print(f"\nRows still intact: {ice.scan().to_arrow().num_rows:,}")

after expiry   snapshots=  3  avro= 40  json= 22  metadata=359.0 KB

Rows still intact: 2,000


### 📝 Nhận xét — `expire_snapshots`

- Giữ 3 snapshot mới nhất: **20 → 3 snapshot** ✅; dữ liệu hiện tại vẫn đủ **2.000 dòng**.
- Nhưng số avro vẫn **40 → 40** và metadata còn **tăng** nhẹ (expiry ghi thêm một `metadata.json` mới: 21 → 22).


### Measured finding: expiry is a **metadata-only** operation

Snapshots dropped from 20 to 3 — but *not one avro file was deleted*, and
metadata on disk actually grew (expiry writes a new `metadata.json`).

This is not a bug. Expiry's job is to make files **unreferenced**. Deleting
them is a *different* job — Job 4. In Java/Spark Iceberg the two are usually
chained; on the Python path you must chain them yourself, or the storage
never shrinks.

In [29]:
print(f"snapshots: {ice_before['snapshots']} → {ice_after['snapshots']}  "
      f"({ice_before['snapshots'] - ice_after['snapshots']} expired)")
print(f"avro files on disk: {ice_before['manifest avro']} → {ice_after['manifest avro']}  "
      f"(deleted: {ice_before['manifest avro'] - ice_after['manifest avro']})")
print(f"metadata bytes: {human(ice_before['metadata bytes'])} → {human(ice_after['metadata bytes'])}")
print("\nExpiry alone reclaimed NOTHING. Now find what it stranded:")

snapshots: 20 → 3  (17 expired)
avro files on disk: 40 → 40  (deleted: 0)
metadata bytes: 350.5 KB → 359.0 KB

Expiry alone reclaimed NOTHING. Now find what it stranded:


In [30]:
def find_iceberg_orphans(table) -> list[Path]:
    """Manifest lists on disk that no live snapshot points at.

    Iceberg names manifest lists `snap-<snapshot-id>-...avro`, and each live
    snapshot records its own in `manifest_list`. The set difference is garbage
    — the same diff `remove_orphan_files` performs.
    """
    live = {Path(s.manifest_list).name for s in table.snapshots()}
    return [f for f in ice_meta.glob("snap-*.avro") if f.name not in live]


stranded = find_iceberg_orphans(ice)
print(f"Stranded manifest lists: {len(stranded)}  ({human(sum(du(f) for f in stranded))})")
for f in stranded[:3]:
    print(f"  {f.name}")
if len(stranded) > 3:
    print(f"  ... and {len(stranded) - 3} more")

reclaimed_ice = sum(du(f) for f in stranded)
for f in stranded:
    f.unlink()

ice_final = ice_metrics("after sweep")
print(f"\nReclaimed by chaining expiry → orphan removal: {human(reclaimed_ice)}")
print(f"Rows still intact: {cat.load_table(f'{ns}.maint').scan().to_arrow().num_rows:,}")
print("\n→ Job 3 and Job 4 are a PAIR. Running expiry without a sweep is why")
print("  teams report 'we expire snapshots but the S3 bill never drops'.")

Stranded manifest lists: 17  (37.3 KB)
  snap-1432531781516856912-0-8ef28b97-a456-4444-9ffb-a8852ad684e7.avro
  snap-1973532168172970087-0-aa6d69dd-6edb-4d00-a898-c4b671f79cf0.avro
  snap-3187212164613361859-0-d6c94b46-3a6d-4024-a488-17b7497081a9.avro
  ... and 14 more


after sweep    snapshots=  3  avro= 23  json= 22  metadata=321.7 KB

Reclaimed by chaining expiry → orphan removal: 37.3 KB
Rows still intact: 2,000

→ Job 3 and Job 4 are a PAIR. Running expiry without a sweep is why
  teams report 'we expire snapshots but the S3 bill never drops'.


### 📝 Nhận xét — Expiry chỉ sửa metadata: Job 3 và Job 4 là một cặp

- Expiry chỉ làm 17 snapshot cũ **không còn được tham chiếu**; **không xoá file vật lý nào** (avro 40 → 40). Đây là hành vi của API `expire_snapshots` trong **phiên bản PyIceberg của lab**, không phải kết luận cho mọi engine Iceberg (Java/Spark thường xoá file luôn khi expire).
- Phép hiệu tập hợp *`snap-*.avro` trên đĩa* − *`manifest_list` của snapshot còn sống* tìm ra **17 manifest list bị bỏ lại** (~37 KB) — đúng bằng số snapshot đã expire. Xoá chúng: avro **40 → 23** (= 3 manifest list còn sống + 20 manifest — vẫn được snapshot mới nhất tham chiếu do *fast append*), dữ liệu vẫn **2.000 dòng** ✅.
- Phân biệt hai trạng thái: **không còn được tham chiếu** (kết quả của expiry) ≠ **đã xoá vật lý** (kết quả của orphan removal). Chỉ chạy expiry mà không dọn file là lý do các đội than "đã expire snapshot mà hoá đơn S3 không giảm".
- Vẫn còn 22 `metadata.json` cũ chưa dọn — một loại rác khác, quản lý bằng `write.metadata.delete-after-commit.enabled` / `write.metadata.previous-versions-max`.


## Who runs these jobs? Self-managed vs managed

"Fully managed" ≠ free. Managed compaction meters **per GB processed** *and*
**per 1,000 objects** — a table with pathological small files is exactly the
table that is most expensive to have auto-compacted.

In [31]:
TABLE_GB, FILES, COMPACTIONS_PER_MONTH = 500, 2_000_000, 30
PRICE_GB, PRICE_PER_1K_OBJ = 0.05, 0.004

gb_cost = TABLE_GB * PRICE_GB * COMPACTIONS_PER_MONTH
obj_cost = FILES / 1000 * PRICE_PER_1K_OBJ * COMPACTIONS_PER_MONTH
print(f"Managed compaction, {TABLE_GB} GB / {FILES:,} files, daily:")
print(f"  per-GB component:     ${gb_cost:,.0f}/mo")
print(f"  per-object component: ${obj_cost:,.0f}/mo")
print(f"  TOTAL:                ${gb_cost + obj_cost:,.0f}/mo")
print(f"\nThe object component is {obj_cost / (gb_cost + obj_cost) * 100:.0f}% of the bill —")
print("it is driven by FILE COUNT, not data volume. Fixing your writer's")
print("trigger interval is cheaper than paying someone to clean up after it.")

Managed compaction, 500 GB / 2,000,000 files, daily:
  per-GB component:     $750/mo
  per-object component: $240/mo
  TOTAL:                $990/mo

The object component is 24% of the bill —
it is driven by FILE COUNT, not data volume. Fixing your writer's
trigger interval is cheaper than paying someone to clean up after it.


### 📝 Nhận xét — Ai chạy các job này: tự quản lý hay managed?

- Compaction managed cho bảng 500 GB / 2 triệu file, chạy hằng ngày: **$750/tháng** theo GB + **$240/tháng** theo số object = **$990/tháng**; phần theo object chiếm **24%** và tỉ lệ với **số file**, không phải dung lượng.
- Bảng có nhiều file nhỏ nhất lại là bảng **đắt nhất khi được auto-compact**. Sửa trigger interval / batch size của writer (ghi ít file hơn từ đầu) rẻ hơn trả tiền để dọn dẹp sau.


## ✅ NB6 pass criteria

| Check | Target |
|---|---|
| Compaction | ≥ 10× fewer files |
| Clustering | ≥ 50% of files skippable for a point query |
| Snapshot expiry | Delta reclaims bytes; Iceberg drops to 3 snapshots |
| Orphan removal | 3 planted Delta orphans + all stranded Iceberg manifest lists swept |
| Log checkpoint | `*.checkpoint.parquet` + `_last_checkpoint` exist |

In [32]:
checks = {
    "compaction ≥ 10x fewer files": base["data files"] / max(after_compact["data files"], 1) >= 10,
    "clustering skips ≥ 50% files": (1 - after_cluster / max(total_files, 1)) >= 0.5,
    "vacuum reclaimed bytes":       before_vacuum > du(TABLE),
    "3 delta orphans removed":      len(found) == 3,
    "no delta orphans remain":      find_orphans(TABLE) == [],
    "checkpoint written":           bool(ckpt) and (log_dir / "_last_checkpoint").exists(),
    "iceberg expired to 3 snaps":   len(ice.snapshots()) == KEEP_LAST,
    "iceberg stranded files swept": len(stranded) > 0 and find_iceberg_orphans(ice) == [],
    "iceberg data intact":          cat.load_table(f"{ns}.maint").scan().to_arrow().num_rows == 2000,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB6 incomplete — see FAIL rows above"
print("\nNB6 complete.")

  [PASS] compaction ≥ 10x fewer files
  [PASS] clustering skips ≥ 50% files
  [PASS] vacuum reclaimed bytes
  [PASS] 3 delta orphans removed
  [PASS] no delta orphans remain
  [PASS] checkpoint written
  [PASS] iceberg expired to 3 snaps
  [PASS] iceberg stranded files swept
  [PASS] iceberg data intact

NB6 complete.


### 🎯 Kết luận NB6

Small-file problem sinh ra từ ingestion bình thường (200 file ~51 KB). Bốn job bảo trì là **bắt buộc** và mỗi job giải quyết một phần khác nhau: **compaction** giảm số file 18× (nhưng tạm thời tăng dung lượng vì file cũ chỉ bị tombstone), **clustering** làm stats hữu ích (skip 90%), **vacuum** thu hồi 16,1 MB đổi lại mất khả năng time travel, **orphan removal** dọn rác mà vacuum của delta-rs không nhìn thấy (3 file chưa từng commit) bằng phép hiệu tập hợp có ngưỡng tuổi; **checkpoint** rút ngắn replay log. Phía Iceberg, `expire_snapshots` của PyIceberg chỉ làm file **không còn được tham chiếu** (20 → 3 snapshot, 0 file bị xoá) — phải chạy thêm bước dọn 17 manifest list. Thông điệp chung: **không còn được tham chiếu ≠ đã xoá vật lý**, và phải kiểm chứng hành vi của đúng engine/phiên bản đang dùng thay vì tin vào slide hay dòng output in sẵn.
